# Agriculture & Climate SLM — Advanced Pipeline

This notebook extends the official starter with:
- Team Kulal curated corpus (competition documents + CGIAR-sourced + FAO-sourced documents), merged and deduplicated.
- LLM-generated `train_qa` rows from curated corpus (with template based on the competition's format)
- A 'metadata-filtered + sentence-transformer' retrieval pipeline (with an optional FAISS backend), validated with Recall@1 / Recall@3
- LoRA fine-tuning using the **official prompt template** (`Crop | Zone | Topic / Question / Context / Answer`) so outputs stay compatible with grading
- Evaluation using **mean Levenshtein distance** — the actual competition metric — comparing retrieval-only vs. fine-tuned RAG
- The same Kaggle-safe submission format checks as the starter notebook

> Run top to bottom on Kaggle with a GPU accelerator and an attached model. Local runs work for development but the graded run must be a committed Kaggle notebook.


## 1. Loading the competition files + installing dependencies

On Kaggle, data is read-only under `/kaggle/input/`. Generated files go to `/kaggle/working/`.

- `kulal_documents.csv` — reference corpus (competition documents + CGIAR-sourced + FAO-sourced documents)

- `kulal_train_qa.csv` — labelled training data (competition examples + llm-generrated examples, extracted from team-curated corpus)

- `test_questions.csv` — competition's hidden test inputs

- `kulal_doc_embeddings.csv` —  embeddings of the reference corpus (competition documents + CGIAR-sourced + FAO-sourced documents obtained from the 'sentence-transformer' option in Step 2.


In [1]:
!pip install -U "transformers>=4.57.1,<5.0.0" "bitsandbytes>=0.46.1" peft trl accelerate python-Levenshtein sentencepiece --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.4 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

TEAM_KULAL_SLUG = "team_kulal_data"
ON_KAGGLE = Path("/kaggle/input").exists()

def find_kulal_data_dir(dataset_slug: str) -> Path:
    
    """Locate 'team Kulal' CSVs under /kaggle/input (layout varies slightly)."""
    
    if not ON_KAGGLE:
        return Path(".")
        
    root = Path("/kaggle/input")
    candidate = root / dataset_slug

    if (candidate / "kulal_train_qa.csv").exists():
        return candidate
        
    for path in root.rglob("kulal_train_qa.csv"):
        return path.parent
        
    raise FileNotFoundError(
        "Could not find the train_qa.csv of your team. Attach your dataset via Add Input."
    )

KULAL_DATA_DIR = find_kulal_data_dir(TEAM_KULAL_SLUG)
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
documents = pd.read_csv(KULAL_DATA_DIR / "kulal_documents.csv")
full_train_qa = pd.read_csv(KULAL_DATA_DIR / "kulal_train_qa.csv")
test = pd.read_csv(KULAL_DATA_DIR / "test_questions.csv")
kulal_embeddings = np.load(KULAL_DATA_DIR / "kulal_doc_embeddings.npy")

print("Kulal's Data dir:", KULAL_DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print(len(documents), "documents ·", len(full_train_qa), "train questions and answers ·", len(test), "test questions")
print("Shape of embeddings :",    kulal_embeddings.shape, "\n\n")
display(full_train_qa.head(3))

Kulal's Data dir: /kaggle/input/datasets/ekaettesamuel/team-kulal-data
Output dir: /kaggle/working
2495 documents · 759 train questions and answers · 12 test questions
Shape of embeddings : (2495, 384) 




,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3


## 1b. Train / validation split

Hold out a slice of `full_train_qa` to evaluate retrieval accuracy and generation quality **before** touching the real test set — this is our own private (hidden) check, similar to the competition's hidden test set.


In [4]:
from sklearn.model_selection import train_test_split

train_qa, val_qa = train_test_split(full_train_qa, test_size=0.15, random_state=42)
print("Train:", len(train_qa), "| Validation:", len(val_qa))

Train: 645 | Validation: 114


## 2. Retrieval baseline (CPU, no fine-tuning)

***Metadata*** (`crop`, `agro_zone`, `topic`) narrows candidates first, ***sentence-transformer embeddings*** then rank
by semantic similarity within that filtered set.

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# This cell was restricted by the 'comment' string.
# You can unrestrict and run cell to obtain embeddings needed for retrieval.
# Saved embeddings run by us (Team_Kulal) was retrieved from the 'DATASET' section of the
# input panel

'''

kulal_embeddings = embedder.encode(
    documents["text"].tolist(),
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True,
)

print("Embedded", kulal_embeddings.shape[0], "documents")

'''

'\n\nkulal_embeddings = embedder.encode(\n    documents["text"].tolist(),\n    convert_to_numpy=True,\n    batch_size=64,\n    show_progress_bar=True,\n)\n\nprint("Embedded", kulal_embeddings.shape[0], "documents")\n\n'

***The Retrieval Function***

In [7]:
def retrieve_document(question, crop, agro_zone, topic=None, top_k=1):
    candidates = documents.copy()

    if pd.notna(crop) and crop != "general":
        crop_match = candidates["crop"].isin([crop, "general"])
        if crop_match.any():
            candidates = candidates[crop_match]

    if pd.notna(agro_zone) and agro_zone != "general":
        zone_match = candidates["agro_zone"].isin([agro_zone, "general"])
        if zone_match.any():
            candidates = candidates[zone_match]

    if topic is not None and pd.notna(topic):
        topic_match = candidates["topic"] == topic
        if topic_match.any():
            candidates = candidates[topic_match]

    if len(candidates) == 0:
        candidates = documents.copy()

    candidate_indices = candidates.index.tolist()
    candidate_embeddings = kulal_embeddings[candidate_indices]

    question_embedding = embedder.encode([question], convert_to_numpy=True)
    similarities = cosine_similarity(question_embedding, candidate_embeddings)[0]

    ranked_idx = np.argsort(similarities)[::-1][:top_k]
    top_doc_indices = [candidate_indices[i] for i in ranked_idx]

    return documents.loc[top_doc_indices]

***Validating retrieval on the held-out validation slice — Recall@1 and Recall@3n***

In [8]:
def evaluate_retrieval(qa_df, top_k=3):
    correct_at_1, correct_at_k = 0, 0
    for _, row in qa_df.iterrows():
        retrieved = retrieve_document(row["question"], row["crop"], row["agro_zone"], row["topic"], top_k=top_k)
        ids = retrieved["document_id"].tolist()
        if ids[0] == row["document_id"]:
            correct_at_1 += 1
        if row["document_id"] in ids:
            correct_at_k += 1
    n = len(qa_df)
    return correct_at_1 / n, correct_at_k / n

recall_1, recall_3 = evaluate_retrieval(val_qa, top_k=3)
print(f"Recall@1: {recall_1:.2%}")
print(f"Recall@3: {recall_3:.2%}")

Recall@1: 87.72%
Recall@3: 93.86%


***Producing answers***

In [9]:
import re

def extract_answer(text, question, embedder, target_chars=120, max_chars=200):
    
    # Merge short fragments (likely abbreviation/decimal splits) into neighbors first
    
    raw_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    sentences = []
    for s in raw_sentences:
        if sentences and len(s) < 15:  # too short to be a real standalone sentence
            sentences[-1] = sentences[-1] + " " + s
        else:
            sentences.append(s)

    if not sentences:
        return text[:max_chars].rsplit(" ", 1)[0]

    # Build candidate spans of 1-2 consecutive sentences, not single sentences alone —
    # keeps context coherent instead of picking an isolated, decontextualized fragment
    
    spans = []
    for i in range(len(sentences)):
        spans.append(sentences[i])
        if i + 1 < len(sentences):
            spans.append(sentences[i] + " " + sentences[i + 1])

    span_embeddings = embedder.encode(spans, convert_to_numpy=True)
    q_embedding = embedder.encode([question], convert_to_numpy=True)
    similarities = cosine_similarity(q_embedding, span_embeddings)[0]

    # Among spans reasonably close to target length, pick the most similar one —
    # avoids picking a 400-char span just because it scored 0.01 higher
    
    length_ok = [i for i, s in enumerate(spans) if len(s) <= max_chars]
    if not length_ok:
        length_ok = list(range(len(spans)))
    best_idx = max(length_ok, key=lambda i: similarities[i])

    return spans[best_idx][:max_chars].strip()

In [10]:
def answer_retrieval_only(question, crop, agro_zone, topic):
    retrieved = retrieve_document(question, crop, agro_zone, topic, top_k=1)
    doc_text = retrieved.iloc[0]["text"]
    return extract_answer(doc_text, question, embedder)

Performing own ***Levenshtein*** scoring

In [11]:
import Levenshtein

answers = [
    extract_answer(retrieve_document(row["question"], row["crop"], row["agro_zone"], row["topic"], top_k=1).iloc[0]["text"], row["question"], embedder)
    for _, row in val_qa.iterrows()
]
distances = [Levenshtein.distance(p, r) for p, r in zip(answers, val_qa["reference_answer"])]
print(f"Mean Levenshtein : {sum(distances)/len(distances):.2f}")

Mean Levenshtein : 117.25


In [12]:
predictions = []
for _, row in test.iterrows():
    baseline_submission_answer = answer_retrieval_only(row["question"], row["crop"], row["agro_zone"], row["topic"])
    predictions.append({"QuestionId": row["QuestionId"], "Answer": baseline_submission_answer})

baseline_submission = pd.DataFrame(predictions)
display(baseline_submission.head())

,QuestionId,Answer
0,1001,Split nitrogen application at planting and kne...
1,1002,Harvest surplus forage at boot stage and sun-d...
2,1003,Bean rust appears as small reddish-brown pustu...
3,1004,"Until now, milk‐producing farms in the peri‐ur..."
4,1005,Apply five to ten tonnes per hectare before pl...


## 3. Validate and save the submission file

Kaggle expects exactly `QuestionId,Answer` — one row per test item, same order as `test_*`.

Writing to **`/kaggle/working/`** on Kaggle (`/kaggle/input` is read-only).


In [13]:
assert list(baseline_submission.columns) == ["QuestionId", "Answer"]
assert len(baseline_submission) == len(test)
assert baseline_submission["QuestionId"].tolist() == test["QuestionId"].tolist()
assert baseline_submission["QuestionId"].is_unique
assert baseline_submission["Answer"].notna().all()

output_path = OUTPUT_DIR / "baseline_submission.csv"
baseline_submission.to_csv(output_path, index=False)
print("Validated and saved", len(baseline_submission), "rows →", output_path)

Validated and saved 12 rows → /kaggle/working/baseline_submission.csv


## 4. Building SFT Examples using retrieved context (not gold documents)

This matters — training on retrieval output (not the "correct" document) means the model learns under the same conditions it'll face at test time, since your test set has no gold ***document_id*** to cheat with.

In [14]:
# Taking out a few xamples that produce terse answers
# This is for the model to learn from

demo_examples = train_qa[train_qa["question"].isin([
    "How many cassava clones were evaluated in the study?",
    "What percentage of bean farmers produce beans before they have identified buyers?",
    "What proportion of farmers grow cassava in Uganda?",
    'How many households in Malawi received remittances in 2016/17?'
])]

print(demo_examples[["question", "crop", "agro_zone", "topic", "reference_answer"]])
print(len(demo_examples))

                                              question     crop  agro_zone  \
541  How many households in Malawi received remitta...    maize  sub_humid   
303  What proportion of farmers grow cassava in Uga...  cassava  sub_humid   
756  What percentage of bean farmers produce beans ...    beans   highland   
661  How many cassava clones were evaluated in the ...  cassava  sub_humid   

             topic                                   reference_answer  
541        general  About half of Malawian households received rem...  
303        general                 About 60% of farmers grow cassava.  
756        general  94% of the farmers produced beans before ident...  
661  crop_diseases                         231 clones were evaluated.  
4


In [15]:
# inference-time demo-q&a for the model to learn from

def build_fewshot_prefix(demo_examples, context_chars=250):
    prefix = ""
    for _, ex in demo_examples.iterrows():
        retrieved = retrieve_document(ex["question"], ex["crop"], ex["agro_zone"], ex["topic"], top_k=1)
        context = retrieved.iloc[0]["text"][:context_chars]
        prefix += (
            f"Crop: {ex['crop']} | Zone: {ex['agro_zone']} | Topic: {ex['topic']}\n"
            f"Question: {ex['question']}\n"
            f"Context: {context}\n"
            f"Answer in ONE concise, direct sentence: {ex['reference_answer']}\n\n"
        )
    return prefix

FEWSHOT_PREFIX = build_fewshot_prefix(demo_examples)
print(FEWSHOT_PREFIX)  # eyeball it before using

Crop: maize | Zone: sub_humid | Topic: general
Question: How many households in Malawi received remittances in 2016/17?
Context: IFPRI Key Facts Series: Social Safety Nets August 2018 Highlights • The percentage of households benefiting from formal social safety nets increased from 17 percent in 2010/11 to 36 percent in 2016/17. • In 2016/17, half of Malawian households receiv
Answer in ONE concise, direct sentence: About half of Malawian households received remittances in 2016/17.

Crop: cassava | Zone: sub_humid | Topic: general
Question: What proportion of farmers grow cassava in Uganda?
Context: Reducing both hunger and high expenditure on food imports is a priority for most developing African countries. Countries that hitherto have relied heavily on food imports are seeking new approaches to increase the utilization of locally grown crops. 
Answer in ONE concise, direct sentence: About 60% of farmers grow cassava.

Crop: beans | Zone: highland | Topic: general
Question: What perce

In [16]:
def build_prompt(question, crop, agro_zone, topic, context_text, context_chars=400):
    return (
        FEWSHOT_PREFIX +
        f"Crop: {crop} | Zone: {agro_zone} | Topic: {topic}\n"
        f"Question: {question}\n"
        f"Context: {context_text[:context_chars]}\n"
        f"Answer in ONE concise, direct sentence:"
    )

sft_rows = []
for _, row in train_qa.iterrows():
    retrieved = retrieve_document(row["question"], row["crop"], row["agro_zone"], row["topic"], top_k=1)
    context = retrieved.iloc[0]["text"]
    prompt = build_prompt(row["question"], row["crop"], row["agro_zone"], row["topic"], context)
    sft_rows.append({"text": prompt + " " + row["reference_answer"]})

print(f"Built {len(sft_rows)} SFT examples")
print(sft_rows[0]["text"])  # eyeball one before training

Built 645 SFT examples
Crop: maize | Zone: sub_humid | Topic: general
Question: How many households in Malawi received remittances in 2016/17?
Context: IFPRI Key Facts Series: Social Safety Nets August 2018 Highlights • The percentage of households benefiting from formal social safety nets increased from 17 percent in 2010/11 to 36 percent in 2016/17. • In 2016/17, half of Malawian households receiv
Answer in ONE concise, direct sentence: About half of Malawian households received remittances in 2016/17.

Crop: cassava | Zone: sub_humid | Topic: general
Question: What proportion of farmers grow cassava in Uganda?
Context: Reducing both hunger and high expenditure on food imports is a priority for most developing African countries. Countries that hitherto have relied heavily on food imports are seeking new approaches to increase the utilization of locally grown crops. 
Answer in ONE concise, direct sentence: About 60% of farmers grow cassava.

Crop: beans | Zone: highland | Topic: gener

## 5. Attaching model 

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

LOCAL_ONLY = True
MODEL_PATH = Path("/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2")


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=LOCAL_ONLY)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=LOCAL_ONLY,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 6. Apply LoRA

**LoRA** trains only low-rank adapter weights (~1–2% of parameters) — faster and lighter than full SFT.

In [18]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


### Train

In [19]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_dataset = Dataset.from_dict({"text": [r["text"] for r in sft_rows]})

training_args = SFTConfig(
    output_dir="/kaggle/working/lora-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    max_length=512,
    dataset_text_field="text",
    fp16=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    loss_type="nll",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

model.save_pretrained("/kaggle/working/lora-adapter-final")
tokenizer.save_pretrained("/kaggle/working/lora-adapter-final")

Adding EOS to train dataset:   0%|          | 0/645 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/645 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/645 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/645 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/645 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
10,1.708900
20,0.716800
30,0.512100
40,0.473800
50,0.471400
60,0.450600
70,0.455800
80,0.459800
90,0.430400
100,0.421300


('/kaggle/working/lora-adapter-final/tokenizer_config.json',
 '/kaggle/working/lora-adapter-final/special_tokens_map.json',
 '/kaggle/working/lora-adapter-final/chat_template.jinja',
 '/kaggle/working/lora-adapter-final/tokenizer.model',
 '/kaggle/working/lora-adapter-final/added_tokens.json',
 '/kaggle/working/lora-adapter-final/tokenizer.json')

### Generate

Short output, matching reference length

In [20]:
model.config.use_cache = True

In [21]:
generation_config = {
    "max_new_tokens": 35,
    "do_sample": False,
    "num_beams": 4,
    "early_stopping": True,
    "repetition_penalty": 1.0,
    "length_penalty": 0.8,
    "pad_token_id": tokenizer.eos_token_id,
}

In [22]:
def generate_finetuned_answer(question, crop, agro_zone, topic, model, tokenizer):
    retrieved = retrieve_document(question, crop, agro_zone, topic, top_k=1)
    context = retrieved.iloc[0]["text"]
    prompt = build_prompt(question, crop, agro_zone, topic, context)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, **generation_config)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("sentence:")[-1].strip()

## 7. Evaluation

### Check for Levenshtein mean

In [23]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

In [24]:
distances = []
for i, (_, row) in enumerate(val_qa.iterrows()):
    pred = generate_finetuned_answer(row["question"], row["crop"], row["agro_zone"], row["topic"], model, tokenizer)
    dist = Levenshtein.distance(pred, row["reference_answer"])
    distances.append(dist)
    print(f"[{i+1}/{len(val_qa)}] distance={dist}")  # <-- shows progress row by row

print(f"\nFine-tuned mean Levenshtein: {sum(distances)/len(distances):.2f}")

[1/114] distance=90
[2/114] distance=153
[3/114] distance=75
[4/114] distance=95
[5/114] distance=153
[6/114] distance=71
[7/114] distance=99
[8/114] distance=27
[9/114] distance=82
[10/114] distance=37
[11/114] distance=90
[12/114] distance=120
[13/114] distance=52
[14/114] distance=81
[15/114] distance=131
[16/114] distance=84
[17/114] distance=98
[18/114] distance=115
[19/114] distance=62
[20/114] distance=65
[21/114] distance=35
[22/114] distance=112
[23/114] distance=79
[24/114] distance=151
[25/114] distance=102
[26/114] distance=111
[27/114] distance=41
[28/114] distance=14
[29/114] distance=36
[30/114] distance=105
[31/114] distance=113
[32/114] distance=156
[33/114] distance=54
[34/114] distance=80
[35/114] distance=33
[36/114] distance=75
[37/114] distance=66
[38/114] distance=170
[39/114] distance=76
[40/114] distance=123
[41/114] distance=159
[42/114] distance=85
[43/114] distance=71
[44/114] distance=124
[45/114] distance=67
[46/114] distance=130
[47/114] distance=75
[48/1

### Build submission answers

In [25]:
predictions = []
for _, row in test.iterrows():
    submission_answer = generate_finetuned_answer(
        row["question"], row["crop"], row["agro_zone"], row["topic"], model, tokenizer
    )
    predictions.append({"QuestionId": row["QuestionId"], "Answer": submission_answer})

lora_submission = pd.DataFrame(predictions)
display(lora_submission.head())

,QuestionId,Answer
0,1001,Split nitrogen application at planting and kne...
1,1002,Harvest surplus forage at boot stage and sun-d...
2,1003,Remove infected debris and avoid overhead irri...
3,1004,"Yes, fresh cow dung on vegetable beds can be a..."
4,1005,Apply five to ten tonnes per hectare before pl...


## 8. Validate final submission and save.

In [26]:
assert list(lora_submission.columns) == ["QuestionId", "Answer"]
assert len(lora_submission) == len(test)
assert lora_submission["QuestionId"].tolist() == test["QuestionId"].tolist()
assert lora_submission["QuestionId"].is_unique
assert lora_submission["Answer"].notna().all()

output_path = OUTPUT_DIR / "lora_submission.csv"
lora_submission.to_csv(output_path, index=False)
print("Validated and saved", len(lora_submission), "rows ->", output_path)
lora_submission.head()

Validated and saved 12 rows -> /kaggle/working/lora_submission.csv


,QuestionId,Answer
0,1001,Split nitrogen application at planting and kne...
1,1002,Harvest surplus forage at boot stage and sun-d...
2,1003,Remove infected debris and avoid overhead irri...
3,1004,"Yes, fresh cow dung on vegetable beds can be a..."
4,1005,Apply five to ten tonnes per hectare before pl...
